In [3]:
# -*- coding: utf-8 -*-
"""
stcis.go.kr Open API (지역코드) 전체 수집 스크립트
- 전국 시도/시군구/읍면동 전부 수집
- 결과물: sido.csv, sigungu.csv, emd.csv, hierarchy.csv
- Python 3.8 호환
"""

from typing import Optional, Dict, Any, List, Tuple
import os
import time
import json
import math
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL = "https://stcis.go.kr/openapi/areacode.json"
API_KEY  = os.getenv("STCIS_API_KEY", "20250814212211uibrf84hdbkrc4sa88jrobb9me")  # <- 환경변수 or 교체

# ====== 설정 ======
TIMEOUT_SEC = 8
MAX_RETRY   = 4                  # 요청 실패시 재시도 횟수
BACKOFF_BASE= 0.7                # 지수 백오프 기준
REQ_DELAY   = 0.10               # 호출 간 딜레이(서버 보호/레이트리밋 회피)
WORKERS_SGG = 10                 # 시군구 병렬 수집 스레드 수
WORKERS_EMD = 16                 # 읍면동 병렬 수집 스레드 수

# 캐시 (중간 저장으로 API 호출 최소화)
CACHE_DIR   = "./stcis_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
CACHE_SIDO  = os.path.join(CACHE_DIR, "sido.json")
CACHE_SGG   = os.path.join(CACHE_DIR, "sigungu.jsonl")  # line-delimited JSON
CACHE_EMD   = os.path.join(CACHE_DIR, "emd.jsonl")

class AreaCodeAPIError(Exception):
    def __init__(self, message: str, payload: Optional[Dict[str, Any]] = None):
        super().__init__(message)
        self.payload = payload or {}

def _validate_params(sdCd: Optional[str], sggCd: Optional[str]) -> None:
    if sdCd is not None:
        if not (isinstance(sdCd, str) and len(sdCd) == 2 and sdCd.isdigit()):
            raise ValueError("sdCd는 2자리 숫자 문자열이어야 합니다. 예) '11'")
    if sggCd is not None:
        if not (isinstance(sggCd, str) and len(sggCd) == 5 and sggCd.isdigit()):
            raise ValueError("sggCd는 5자리 숫자 문자열이어야 합니다. 예) '11110'")

def _request_once(apikey: str, sdCd: Optional[str], sggCd: Optional[str]) -> Dict[str, Any]:
    params = {"apikey": apikey}
    if sdCd:  params["sdCd"]  = sdCd
    if sggCd: params["sggCd"] = sggCd
    r = requests.get(BASE_URL, params=params, timeout=TIMEOUT_SEC)
    r.raise_for_status()
    data = r.json()
    status = data.get("status")
    if status == "OK" or status == "NOT_FOUND":
        return data
    # status == ERROR 등
    err = data.get("error", {})
    msg = f"API 오류(status={status}): [{err.get('code')}] {err.get('text')} (level={err.get('level')})"
    raise AreaCodeAPIError(msg, payload=data)

def _request(apikey: str, sdCd: Optional[str]=None, sggCd: Optional[str]=None) -> Dict[str, Any]:
    _validate_params(sdCd, sggCd)
    for i in range(MAX_RETRY + 1):
        try:
            data = _request_once(apikey, sdCd, sggCd)
            # 약한 레이트리밋 회피
            time.sleep(REQ_DELAY)
            return data
        except (requests.RequestException, AreaCodeAPIError) as e:
            if i >= MAX_RETRY:
                raise
            # 지수 백오프
            sleep_s = BACKOFF_BASE * (2 ** i) + (0.05 * i)
            time.sleep(sleep_s)

def _normalize_result(data: Dict[str, Any]) -> List[Dict[str, Any]]:
    if data.get("status") == "NOT_FOUND":
        return []
    result = data.get("result")
    if result is None:
        return []
    if isinstance(result, list):
        return result
    if isinstance(result, dict):
        return [result]
    return []

# 단일 호출 래퍼(DF 반환)
def list_sido(apikey: str) -> pd.DataFrame:
    data = _request(apikey)
    rows = _normalize_result(data)
    df = pd.json_normalize(rows)
    # 기대 컬럼: sdCd, sdNm
    if not df.empty:
        df = df.rename(columns={"sdCd":"시도코드","sdNm":"시도명"})
        df = df.sort_values(["시도코드"]).reset_index(drop=True)
    return df

def list_sigungu(apikey: str, sdCd: str) -> pd.DataFrame:
    data = _request(apikey, sdCd=sdCd)
    rows = _normalize_result(data)
    df = pd.json_normalize(rows)
    # 기대 컬럼: sggCd, sggNm
    if not df.empty:
        df["시도코드"] = sdCd
        df = df.rename(columns={"sggCd":"시군구코드","sggNm":"시군구명"})
        df = df[["시도코드","시군구코드","시군구명"]].sort_values(["시도코드","시군구코드"]).reset_index(drop=True)
    return df

def list_emd(apikey: str, sggCd: str) -> pd.DataFrame:
    data = _request(apikey, sggCd=sggCd)
    rows = _normalize_result(data)
    df = pd.json_normalize(rows)
    # 기대 컬럼: emdCd, emdNm
    if not df.empty:
        df["시군구코드"] = sggCd
        df = df.rename(columns={"emdCd":"읍면동코드","emdNm":"읍면동명"})
        df = df[["시군구코드","읍면동코드","읍면동명"]].sort_values(["시군구코드","읍면동코드"]).reset_index(drop=True)
    return df

# ====== 전체 수집 ======
def fetch_all(apikey: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # 1) 시도
    if os.path.exists(CACHE_SIDO):
        with open(CACHE_SIDO, "r", encoding="utf-8") as f:
            sido_rows = json.load(f)
        df_sido = pd.json_normalize(sido_rows)
        if not df_sido.empty:
            df_sido = df_sido.rename(columns={"sdCd":"시도코드","sdNm":"시도명"}).sort_values("시도코드").reset_index(drop=True)
    else:
        df_sido = list_sido(apikey)
        # 캐시 저장(원형 유지)
        rows = [] if df_sido.empty else [{"sdCd": r["시도코드"], "sdNm": r["시도명"]} for _, r in df_sido.iterrows()]
        with open(CACHE_SIDO, "w", encoding="utf-8") as f:
            json.dump(rows, f, ensure_ascii=False, indent=2)
    print(f"[1/3] 시도 {len(df_sido)}건 수집")

    # 2) 시군구
    sgg_rows: List[Dict[str, Any]] = []
    # 캐시 로드
    if os.path.exists(CACHE_SGG):
        with open(CACHE_SGG, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    sgg_rows.append(json.loads(line))
    existing_sgg_keys = {(r["sdCd"], r["sggCd"]) for r in sgg_rows}

    def _fetch_sgg(sdCd: str) -> List[Dict[str, Any]]:
        df = list_sigungu(apikey, sdCd=sdCd)
        if df.empty:
            return []
        out = [{"sdCd": sdCd, "sggCd": r["시군구코드"], "sggNm": r["시군구명"]} for _, r in df.iterrows()]
        return out

    with ThreadPoolExecutor(max_workers=WORKERS_SGG) as ex:
        futures = []
        for sdCd in df_sido["시도코드"].tolist():
            futures.append(ex.submit(_fetch_sgg, sdCd))
        for fut in as_completed(futures):
            try:
                rows = fut.result()
                # 캐시에 append
                with open(CACHE_SGG, "a", encoding="utf-8") as f:
                    for r in rows:
                        key = (r["sdCd"], r["sggCd"])
                        if key in existing_sgg_keys:
                            continue
                        existing_sgg_keys.add(key)
                        f.write(json.dumps(r, ensure_ascii=False) + "\n")
                        sgg_rows.append(r)
            except Exception as e:
                print("[경고] 시군구 수집 실패:", e)

    df_sgg = pd.DataFrame(sgg_rows)
    if not df_sgg.empty:
        df_sgg = df_sgg.rename(columns={"sdCd":"시도코드","sggCd":"시군구코드","sggNm":"시군구명"})
        df_sgg = df_sgg.sort_values(["시도코드","시군구코드"]).reset_index(drop=True)
    print(f"[2/3] 시군구 {len(df_sgg)}건 수집")

    # 3) 읍면동
    emd_rows: List[Dict[str, Any]] = []
    if os.path.exists(CACHE_EMD):
        with open(CACHE_EMD, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    emd_rows.append(json.loads(line))
    existing_emd_keys = {(r["sggCd"], r["emdCd"]) for r in emd_rows}

    def _fetch_emd(sggCd: str) -> List[Dict[str, Any]]:
        df = list_emd(apikey, sggCd=sggCd)
        if df.empty:
            return []
        out = [{"sggCd": sggCd, "emdCd": r["읍면동코드"], "emdNm": r["읍면동명"]} for _, r in df.iterrows()]
        return out

    with ThreadPoolExecutor(max_workers=WORKERS_EMD) as ex:
        futures = []
        for sggCd in df_sgg["시군구코드"].tolist():
            futures.append(ex.submit(_fetch_emd, sggCd))
        for fut in as_completed(futures):
            try:
                rows = fut.result()
                with open(CACHE_EMD, "a", encoding="utf-8") as f:
                    for r in rows:
                        key = (r["sggCd"], r["emdCd"])
                        if key in existing_emd_keys:
                            continue
                        existing_emd_keys.add(key)
                        f.write(json.dumps(r, ensure_ascii=False) + "\n")
                        emd_rows.append(r)
            except Exception as e:
                print("[경고] 읍면동 수집 실패:", e)

    df_emd = pd.DataFrame(emd_rows)
    if not df_emd.empty:
        df_emd = df_emd.rename(columns={"sggCd":"시군구코드","emdCd":"읍면동코드","emdNm":"읍면동명"})
        df_emd = df_emd.sort_values(["시군구코드","읍면동코드"]).reset_index(drop=True)
    print(f"[3/3] 읍면동 {len(df_emd)}건 수집")

    # 4) 계층 조인(hierarchy)
    # 시군구에 시도명 붙이기
    df_sgg_join = df_sgg.merge(df_sido, on="시도코드", how="left")
    # 읍면동에 시군구/시도 붙이기
    df_hier = df_emd.merge(df_sgg_join[["시군구코드","시군구명","시도코드","시도명"]],
                           on="시군구코드", how="left")
    df_hier = df_hier[["시도코드","시도명","시군구코드","시군구명","읍면동코드","읍면동명"]] \
                     .sort_values(["시도코드","시군구코드","읍면동코드"]).reset_index(drop=True)

    return df_sido, df_sgg, df_emd, df_hier

def save_all(df_sido: pd.DataFrame, df_sgg: pd.DataFrame, df_emd: pd.DataFrame, df_hier: pd.DataFrame, out_dir: str = ".") -> None:
    os.makedirs(out_dir, exist_ok=True)
    df_sido.to_csv(os.path.join(out_dir, "sido.csv"), index=False, encoding="utf-8-sig")
    df_sgg.to_csv(os.path.join(out_dir, "sigungu.csv"), index=False, encoding="utf-8-sig")
    df_emd.to_csv(os.path.join(out_dir, "emd.csv"), index=False, encoding="utf-8-sig")
    df_hier.to_csv(os.path.join(out_dir, "hierarchy.csv"), index=False, encoding="utf-8-sig")
    print(f"저장 완료: {out_dir}/sido.csv, sigungu.csv, emd.csv, hierarchy.csv")

if __name__ == "__main__":
    if API_KEY in (None, "", "여기에_API_KEY_넣으세요"):
        raise RuntimeError("환경변수 STCIS_API_KEY 를 설정하거나 코드의 API_KEY 값을 실제 키로 바꾸세요.")
    sido, sgg, emd, hier = fetch_all(API_KEY)
    save_all(sido, sgg, emd, hier, out_dir=".")


[1/3] 시도 25건 수집
[2/3] 시군구 477건 수집
[3/3] 읍면동 49337건 수집
저장 완료: ./sido.csv, sigungu.csv, emd.csv, hierarchy.csv


In [1]:
# -*- coding: utf-8 -*-
"""
sido.csv의 '시도코드' 전부 대상으로 busroute.json 호출
- OUT_CSV에 이미 있는 (API시도코드, 노선번호)는 스킵 → 없는 것만 조회
- 별명/호환코드(49↔50, 51→42 등) 미사용: sido.csv에 적힌 코드만 그대로 호출
"""

from typing import Dict, Any, List, Tuple, Set
import os, time, json, re, threading, copy
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except Exception:
    Retry = None

# ===== 설정 =====
BASE_URL = "https://stcis.go.kr/openapi/busroute.json"
API_KEY  = os.getenv("STCIS_API_KEY", "20250814212211uibrf84hdbkrc4sa88jrobb9me")  # 실제 키로 교체하거나 환경변수 설정

SIDO_CSV = "sido.csv"                 # 입력: 반드시 '시도코드' 열 포함
OUT_CSV  = "bus_routes_all.csv"       # 출력: 증분 기준 파일

TIMEOUT_SEC    = 6
MAX_RETRY      = 2
BACKOFF_BASE   = 0.4
REQ_DELAY      = 0.03
WORKERS        = 64
BATCH_SIZE     = 800
CACHE_PATH     = "./busroute_cache.json"
CACHE_FLUSH_EVERY = 4000

# ===== 전역 락 =====
cache_lock = threading.Lock()
file_lock  = threading.Lock()
print_lock = threading.Lock()

# ===== 세션 =====
def make_session() -> requests.Session:
    s = requests.Session()
    if Retry is not None:
        retry = Retry(
            total=MAX_RETRY,
            backoff_factor=0.2,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"],
            raise_on_status=False,
        )
        adapter = HTTPAdapter(pool_connections=WORKERS*2, pool_maxsize=WORKERS*2, max_retries=retry)
    else:
        adapter = HTTPAdapter(pool_connections=WORKERS*2, pool_maxsize=WORKERS*2)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

# ===== I/O =====
def load_sido_codes(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    if "시도코드" not in df.columns:
        # 보조 컬럼명 허용
        for cand in ["sdCd","sido_code","sd_cd","sdcd"]:
            if cand in df.columns:
                df["시도코드"] = df[cand]
                break
    if "시도코드" not in df.columns:
        raise ValueError("sido.csv에 '시도코드' 열이 없습니다.")
    if "시도명" not in df.columns:
        df["시도명"] = ""

    out = pd.DataFrame({
        "시도코드": df["시도코드"].astype(str).str.extract(r"(\d+)")[0].dropna().str.zfill(2),
        "시도명":   df["시도명"].astype(str)
    }).drop_duplicates(subset=["시도코드"]).sort_values("시도코드").reset_index(drop=True)
    out = out[out["시도코드"].str.fullmatch(r"\d{2}")]
    if out.empty:
        raise ValueError("유효한 2자리 시도코드를 찾지 못했습니다.")
    return out

def load_done_map(out_csv: str) -> Dict[str, Set[str]]:
    """
    OUT_CSV에서 이미 존재하는 (API시도코드, 노선번호) 집합 생성.
    스키마가 조금 달라도 최대한 인식.
    """
    done: Dict[str, Set[str]] = {}
    if not os.path.exists(out_csv):
        return done

    api_col = None
    rn_col  = "노선번호"
    for chunk in pd.read_csv(out_csv, encoding="utf-8-sig", chunksize=200_000, dtype=str):
        cols = set(chunk.columns)
        if api_col is None:
            if "API시도코드" in cols and rn_col in cols:
                api_col = "API시도코드"
            elif "시도코드" in cols and rn_col in cols:
                api_col = "시도코드"
            else:
                print("[WARN] OUT_CSV에서 (API시도코드|시도코드, 노선번호) 컬럼을 찾지 못함 → 증분 비활성.")
                return {}
        chunk[api_col] = chunk[api_col].astype(str).str.zfill(2)
        chunk[rn_col]  = chunk[rn_col].astype(str).str.strip()
        for sd, rn in zip(chunk[api_col], chunk[rn_col]):
            if sd and rn:
                done.setdefault(sd, set()).add(rn)
    total = sum(len(v) for v in done.values())
    print(f"[INFO] 기존 누적: {total}개 (시도 {len(done)})")
    return done

def _load_cache() -> Dict[str, Dict[str, Any]]:
    if not os.path.exists(CACHE_PATH):
        return {}
    try:
        with open(CACHE_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {}

def _save_cache(cache: Dict[str, Dict[str, Any]]) -> None:
    with cache_lock:
        snapshot = copy.deepcopy(cache)
    tmp = CACHE_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(snapshot, f, ensure_ascii=False)
    os.replace(tmp, CACHE_PATH)

def append_rows_to_csv(rows: List[Dict[str, Any]], path: str) -> None:
    if not rows:
        return
    df = pd.DataFrame(rows)
    with file_lock:
        file_exists = os.path.exists(path)
        df.to_csv(path, mode="a", index=False, encoding="utf-8-sig", header=not file_exists)

# ===== API =====
class APIError(Exception): pass

def _request_once(session: requests.Session, sdCd: str, routeNo: str) -> Dict[str, Any]:
    params = {"apikey": API_KEY, "sdCd": sdCd, "routeNo": routeNo}
    r = session.get(BASE_URL, params=params, timeout=TIMEOUT_SEC)
    r.raise_for_status()
    data = r.json()
    status = data.get("status")
    if status in ("OK", "NOT_FOUND"):
        return data
    err = data.get("error", {})
    raise APIError(f"API status={status} [{err.get('code')}] {err.get('text')}")

def _request(session: requests.Session, sdCd: str, routeNo: str) -> Dict[str, Any]:
    for i in range(MAX_RETRY + 1):
        try:
            res = _request_once(session, sdCd, routeNo)
            time.sleep(REQ_DELAY)
            return res
        except Exception:
            if i == MAX_RETRY:
                raise
            time.sleep(BACKOFF_BASE * (2**i))

def _normalize_result(data: Dict[str, Any]) -> List[Dict[str, Any]]:
    if data.get("status") != "OK":
        return []
    result = data.get("result")
    if isinstance(result, dict): return [result]
    if isinstance(result, list): return result
    return []

# ===== 처리 =====
def process_sido(sdCd: str, name: str, session: requests.Session,
                 cache: Dict[str, Dict[str, Any]],
                 done_map: Dict[str, Set[str]]) -> None:
    """
    sido.csv의 코드 그대로 사용.
    OUT_CSV에 이미 있는 (sdCd, routeNo)는 스킵.
    """
    print(f"[START] {sdCd} {name}".strip())
    total = 9999
    skip_set = done_map.get(sdCd, set())
    route_nos = [rn for rn in range(1, total + 1) if str(rn) not in skip_set]
    if not route_nos:
        print(f"[SKIP] {sdCd}: 조회할 노선 없음(모두 기존 CSV에 존재).")
        return

    def _fetch(rn: int) -> Tuple[str, str, Dict[str, Any]]:
        key = f"{sdCd}|{rn}"
        with cache_lock:
            hit = cache.get(key)
        if hit is not None:
            return sdCd, str(rn), hit
        data = _request(session, sdCd, str(rn))
        with cache_lock:
            cache[key] = data
        return sdCd, str(rn), data

    processed = 0
    seen_route_ids: set = set()  # 시도 단위 중복(희박하지만) 방지
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        for start in range(0, len(route_nos), BATCH_SIZE):
            batch = route_nos[start:start+BATCH_SIZE]
            futs = [ex.submit(_fetch, rn) for rn in batch]
            batch_rows: List[Dict[str, Any]] = []
            for fut in as_completed(futs):
                processed += 1
                _, rn, data = fut.result()
                for rowj in _normalize_result(data):
                    rid = str(rowj.get("routeId") or "")
                    if rid and rid in seen_route_ids:
                        continue
                    if rid:
                        seen_route_ids.add(rid)
                    batch_rows.append({
                        "API시도코드": sdCd,
                        "시도명": name,
                        "노선번호": rowj.get("routeNo", rn),
                        "routeId":  rowj.get("routeId"),
                        "기점":     rowj.get("stgSttnNma"),
                        "종점":     rowj.get("arrSttnNma"),
                    })
                if processed % CACHE_FLUSH_EVERY == 0:
                    _save_cache(cache)
                    print(f"[{sdCd}] progress: {processed}/{len(route_nos)} (cache flushed)")

            if batch_rows:
                append_rows_to_csv(batch_rows, OUT_CSV)

            pct = processed * 100.0 / max(1, len(route_nos))
            print(f"[{sdCd}] batch {start+1}-{start+len(batch)} done: {processed}/{len(route_nos)} ({pct:.1f}%)")

    _save_cache(cache)
    print(f"[{sdCd}] completed. unique routes added={len(seen_route_ids)}")

def main():
    if API_KEY in (None, "", "여기에_API_KEY_넣으세요", "YOUR_API_KEY", "your_api_key_here"):
        raise RuntimeError("환경변수 STCIS_API_KEY 설정 또는 코드의 API_KEY를 실제 키로 교체하세요.")

    sido_df  = load_sido_codes(SIDO_CSV)        # 시도코드만 그대로 사용
    done_map = load_done_map(OUT_CSV)           # 증분 기준
    cache    = _load_cache()
    session  = make_session()

    for _, r in sido_df.iterrows():
        process_sido(r["시도코드"], r.get("시도명",""), session, cache, done_map)

    print(f"[DONE] 증분 수집 완료 → {OUT_CSV}")

if __name__ == "__main__":
    main()


[INFO] 기존 누적: 4093개 (시도 16)
[START] 11 서울특별시
[11] batch 1-800 done: 800/9629 (8.3%)
[11] batch 801-1600 done: 1600/9629 (16.6%)
[11] batch 1601-2400 done: 2400/9629 (24.9%)
[11] batch 2401-3200 done: 3200/9629 (33.2%)
[11] progress: 4000/9629 (cache flushed)
[11] batch 3201-4000 done: 4000/9629 (41.5%)
[11] batch 4001-4800 done: 4800/9629 (49.8%)
[11] batch 4801-5600 done: 5600/9629 (58.2%)
[11] batch 5601-6400 done: 6400/9629 (66.5%)
[11] batch 6401-7200 done: 7200/9629 (74.8%)
[11] progress: 8000/9629 (cache flushed)
[11] batch 7201-8000 done: 8000/9629 (83.1%)
[11] batch 8001-8800 done: 8800/9629 (91.4%)
[11] batch 8801-9600 done: 9600/9629 (99.7%)
[11] batch 9601-9629 done: 9629/9629 (100.0%)
[11] completed. unique routes added=0
[START] 21 부산직할시
[21] batch 1-800 done: 800/9999 (8.0%)
[21] batch 801-1600 done: 1600/9999 (16.0%)
[21] batch 1601-2400 done: 2400/9999 (24.0%)
[21] batch 2401-3200 done: 3200/9999 (32.0%)
[21] progress: 4000/9999 (cache flushed)
[21] batch 3201-4000 done

In [5]:
# -*- coding: utf-8 -*-
"""
sido.csv '시도코드' 전수 → busroute.json (증분 + 호환코드 + 제주 보강)
- OUT_CSV에 있는 (API시도코드, 노선번호) 스킵
- 49/50(제주), 51(강원특별자치도), 52(전북특별자치도) 호환코드 자동 재시도
- 제주에서 숫자형이 0건이면: 001~999 제로패딩, 100-1..900-1 하이픈 패턴도 시도
"""

from typing import Dict, Any, List, Tuple, Set
import os, time, json, re, threading, copy
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except Exception:
    Retry = None

# ===== 설정 =====
BASE_URL = "https://stcis.go.kr/openapi/busroute.json"
API_KEY  = os.getenv("STCIS_API_KEY", "20250814212211uibrf84hdbkrc4sa88jrobb9me")  # 실제 키 or env

SIDO_CSV = "sido.csv"
OUT_CSV  = "bus_routes_all_2.csv"

TIMEOUT_SEC    = 6
MAX_RETRY      = 2
BACKOFF_BASE   = 0.4
REQ_DELAY      = 0.03
WORKERS        = 64
BATCH_SIZE     = 800
CACHE_PATH     = "./busroute_cache.json"
CACHE_FLUSH_EVERY = 4000

# ===== 호환코드 =====
SDCD_ALIASES: Dict[str, List[str]] = {
    "49": ["49", "50"],   # 제주 혼재
    "50": ["50", "49"],
    "51": ["51", "42"],   # 강원특별자치도 → 강원도
    "52": ["52", "45"],   # 전북특별자치도 → 전라북도
}
JEJU_CODES = {"49", "50"}

def sdcd_try_list(sd: str) -> List[str]:
    sd = str(sd).zfill(2)
    return SDCD_ALIASES.get(sd, [sd])

# ===== 락 =====
cache_lock = threading.Lock()
file_lock  = threading.Lock()
print_lock = threading.Lock()

# ===== 세션 =====
def make_session() -> requests.Session:
    s = requests.Session()
    if Retry is not None:
        retry = Retry(total=MAX_RETRY, backoff_factor=0.2,
                      status_forcelist=[429, 500, 502, 503, 504],
                      allowed_methods=["GET"], raise_on_status=False)
        adapter = HTTPAdapter(pool_connections=WORKERS*2, pool_maxsize=WORKERS*2, max_retries=retry)
    else:
        adapter = HTTPAdapter(pool_connections=WORKERS*2, pool_maxsize=WORKERS*2)
    s.mount("https://", adapter); s.mount("http://", adapter)
    return s

# ===== I/O =====
def load_sido_df(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    code_col = next((c for c in ["시도코드","sdCd","sido_code","sd_cd","sdcd"] if c in df.columns), None)
    name_col = next((c for c in ["시도명","sdNm","sido_name","sd_nm"] if c in df.columns), None)
    if code_col is None:
        raise ValueError("sido.csv에 시도코드 열이 없습니다.")
    if name_col is None:
        df["시도명"] = ""; name_col = "시도명"
    out = pd.DataFrame({
        "시도코드": df[code_col].astype(str).str.extract(r"(\d+)")[0].dropna().str.zfill(2),
        "시도명":   df[name_col].astype(str)
    }).drop_duplicates(subset=["시도코드"]).sort_values("시도코드").reset_index(drop=True)
    out = out[out["시도코드"].str.fullmatch(r"\d{2}")]
    if out.empty: raise ValueError("유효한 2자리 시도코드를 찾지 못했습니다.")
    return out

def load_done_map(out_csv: str) -> Dict[str, Set[str]]:
    done: Dict[str, Set[str]] = {}
    if not os.path.exists(out_csv):
        return done
    api_col = None; rn_col = "노선번호"
    for chunk in pd.read_csv(out_csv, encoding="utf-8-sig", chunksize=200_000, dtype=str):
        cols = set(chunk.columns)
        if api_col is None:
            if "API시도코드" in cols and rn_col in cols: api_col = "API시도코드"
            elif "시도코드" in cols and rn_col in cols:  api_col = "시도코드"
            else:
                print("[WARN] (API시도코드|시도코드, 노선번호) 못찾음 → 증분 비활성"); return {}
        chunk[api_col] = chunk[api_col].astype(str).str.zfill(2)
        chunk[rn_col]  = chunk[rn_col].astype(str).str.strip()
        for sd, rn in zip(chunk[api_col], chunk[rn_col]):
            if sd and rn:
                done.setdefault(sd, set()).add(rn)
    total = sum(len(v) for v in done.values())
    print(f"[INFO] 기존 누적: {total}개 (시도 {len(done)})")
    return done

def _load_cache() -> Dict[str, Dict[str, Any]]:
    if not os.path.exists(CACHE_PATH): return {}
    try:
        with open(CACHE_PATH, "r", encoding="utf-8") as f: return json.load(f)
    except Exception: return {}

def _save_cache(cache: Dict[str, Dict[str, Any]]) -> None:
    with cache_lock: snapshot = copy.deepcopy(cache)
    tmp = CACHE_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f: json.dump(snapshot, f, ensure_ascii=False)
    os.replace(tmp, CACHE_PATH)

def append_rows_to_csv(rows: List[Dict[str, Any]], path: str) -> None:
    if not rows: return
    df = pd.DataFrame(rows)
    with file_lock:
        file_exists = os.path.exists(path)
        df.to_csv(path, mode="a", index=False, encoding="utf-8-sig", header=not file_exists)

# ===== API =====
class APIError(Exception): pass

def _request_once(session: requests.Session, sdCd: str, routeNo: str) -> Dict[str, Any]:
    params = {"apikey": API_KEY, "sdCd": sdCd, "routeNo": routeNo}
    r = session.get(BASE_URL, params=params, timeout=TIMEOUT_SEC)
    r.raise_for_status()
    data = r.json()
    if data.get("status") in ("OK", "NOT_FOUND"):
        return data
    err = data.get("error", {})
    raise APIError(f"API status={data.get('status')} [{err.get('code')}] {err.get('text')}")

def _request(session: requests.Session, sdCd: str, routeNo: str) -> Dict[str, Any]:
    for i in range(MAX_RETRY + 1):
        try:
            res = _request_once(session, sdCd, routeNo)
            time.sleep(REQ_DELAY)
            return res
        except Exception:
            if i == MAX_RETRY: raise
            time.sleep(BACKOFF_BASE * (2**i))

def _normalize_result(data: Dict[str, Any]) -> List[Dict[str, Any]]:
    if data.get("status") != "OK": return []
    res = data.get("result")
    if isinstance(res, dict): return [res]
    if isinstance(res, list): return res
    return []

# ===== 후보 생성 =====
def gen_numeric_candidates(skip: Set[str]) -> List[str]:
    # 1..9999 숫자 문자열
    return [str(n) for n in range(1, 10000) if str(n) not in skip]

def gen_jeju_fallback_candidates(skip: Set[str]) -> List[str]:
    # 제로패딩 3자리(001~999) + 하이픈 100-1..900-1
    cands = []
    cands += [f"{n:03d}" for n in range(1, 1000) if f"{n:03d}" not in skip]
    cands += [f"{h}00-1" if h < 100 else f"{h}-1" for h in range(100, 1000, 100)]
    # 위 하이픈 세트는 100-1,200-1,...,900-1 (중복 제거)
    cands = list(dict.fromkeys([c for c in cands if c not in skip]))
    return cands

# ===== 처리 =====
def process_sido(sdCd: str, name: str, session: requests.Session,
                 cache: Dict[str, Dict[str, Any]],
                 done_map: Dict[str, Set[str]]) -> None:
    sdCd = str(sdCd).zfill(2)
    print(f"[START] {sdCd} {name}".strip())
    seen_route_ids: Set[str] = set()

    def _fetch(api_sd: str, rn: str) -> Tuple[str, str, Dict[str, Any]]:
        key = f"{api_sd}|{rn}"
        with cache_lock:
            hit = cache.get(key)
        if hit is not None:
            return api_sd, rn, hit
        data = _request(session, api_sd, rn)
        with cache_lock:
            cache[key] = data
        return api_sd, rn, data

    # 호환코드 순서대로
    for api_sd in sdcd_try_list(sdCd):
        skip_set = done_map.get(api_sd, set())
        # 1차: 숫자형
        route_nos = gen_numeric_candidates(skip_set)
        ok_count = 0

        def _run_batch(candidates: List[str]) -> int:
            processed = 0
            ok_local = 0
            with ThreadPoolExecutor(max_workers=WORKERS) as ex:
                for start in range(0, len(candidates), BATCH_SIZE):
                    batch = candidates[start:start+BATCH_SIZE]
                    futs = [ex.submit(_fetch, api_sd, rn) for rn in batch]
                    out_rows: List[Dict[str, Any]] = []
                    for fut in as_completed(futs):
                        processed += 1
                        _, rn, data = fut.result()
                        rows = _normalize_result(data)
                        if rows:
                            ok_local += 1
                        for rowj in rows:
                            rid = str(rowj.get("routeId") or "")
                            if rid and rid in seen_route_ids:
                                continue
                            if rid:
                                seen_route_ids.add(rid)
                            out_rows.append({
                                "요청시도코드": sdCd,
                                "API시도코드": api_sd,
                                "시도명": name,
                                "노선번호": rowj.get("routeNo", rn),
                                "routeId":  rowj.get("routeId"),
                                "기점":     rowj.get("stgSttnNma"),
                                "종점":     rowj.get("arrSttnNma"),
                            })
                        if processed % CACHE_FLUSH_EVERY == 0:
                            _save_cache(cache)
                            print(f"[{sdCd}->{api_sd}] progress: {processed}/{len(candidates)} (cache flushed)")
                    if out_rows:
                        append_rows_to_csv(out_rows, OUT_CSV)
                    pct = processed * 100.0 / max(1, len(candidates))
                    print(f"[{sdCd}->{api_sd}] batch {start+1}-{start+len(batch)} done: {processed}/{len(candidates)} ({pct:.1f}%)")
            return ok_local

        ok_count += _run_batch(route_nos)

        # 2차: 제주 보강(숫자형에서 0건일 때만)
        if ok_count == 0 and api_sd in JEJU_CODES:
            print(f"[{sdCd}->{api_sd}] 숫자형 0건 → 제주 보강 후보(제로패딩/하이픈) 시도")
            extra = gen_jeju_fallback_candidates(skip_set)
            ok_count += _run_batch(extra)

        # 이 alias에서 뭔가라도 수확했으면 다음 alias는 중복만 늘 가능성 ↑ → 종료
        if seen_route_ids or ok_count > 0:
            print(f"[{sdCd}] collected {len(seen_route_ids)} unique routes via sdCd={api_sd}. Stop trying aliases.")
            break

    _save_cache(cache)
    print(f"[{sdCd}] completed. unique routes added={len(seen_route_ids)}")

def main():
    if API_KEY in (None, "", "여기에_API_KEY_넣으세요", "YOUR_API_KEY", "your_api_key_here"):
        raise RuntimeError("환경변수 STCIS_API_KEY 설정 또는 코드의 API_KEY를 실제 키로 교체하세요.")

    sido_df  = load_sido_df(SIDO_CSV)
    done_map = load_done_map(OUT_CSV)
    cache    = _load_cache()
    session  = make_session()

    for _, r in sido_df.iterrows():
        process_sido(r["시도코드"], r.get("시도명",""), session, cache, done_map)

    print(f"[DONE] 증분 수집 완료 → {OUT_CSV}")

if __name__ == "__main__":
    main()


[INFO] 기존 누적: 4093개 (시도 16)
[START] 11 서울특별시
[11->11] batch 1-800 done: 800/9629 (8.3%)
[11->11] batch 801-1600 done: 1600/9629 (16.6%)
[11->11] batch 1601-2400 done: 2400/9629 (24.9%)
[11->11] batch 2401-3200 done: 3200/9629 (33.2%)
[11->11] progress: 4000/9629 (cache flushed)
[11->11] batch 3201-4000 done: 4000/9629 (41.5%)
[11->11] batch 4001-4800 done: 4800/9629 (49.8%)
[11->11] batch 4801-5600 done: 5600/9629 (58.2%)
[11->11] batch 5601-6400 done: 6400/9629 (66.5%)
[11->11] batch 6401-7200 done: 7200/9629 (74.8%)
[11->11] progress: 8000/9629 (cache flushed)
[11->11] batch 7201-8000 done: 8000/9629 (83.1%)
[11->11] batch 8001-8800 done: 8800/9629 (91.4%)
[11->11] batch 8801-9600 done: 9600/9629 (99.7%)
[11->11] batch 9601-9629 done: 9629/9629 (100.0%)
[11] completed. unique routes added=0
[START] 21 부산직할시
[21->21] batch 1-800 done: 800/9999 (8.0%)
[21->21] batch 801-1600 done: 1600/9999 (16.0%)
[21->21] batch 1601-2400 done: 2400/9999 (24.0%)
[21->21] batch 2401-3200 done: 3200/999

In [1]:
# -*- coding: utf-8 -*-
"""
busroutesttn(노선별 경유정류장) 수집 - 강원/전북/제주 보강 + numpy 난수 고정
- hierarchy.csv(시군/읍면동 코드) + bus_routes_all_2.csv(routeId) 사용
- 51↔42, 52↔45, 49↔50 코드 혼재 자동 처리
- sdCd 실패 시 sgg/emd 폭넓은 재조회(노선별 고정 시드로 섞어 커버)
- 결과: routeId, 버스번호, 정류장순번, 정류장ID, 정류장명, 시도, 시군구, 읍면동
"""

import os, time, json, re, threading, copy, uuid, hashlib
from typing import Dict, Any, List, Tuple, Set, Optional
import requests
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except Exception:
    Retry = None

# ===================== 설정 =====================
API_KEY   = os.getenv("STCIS_API_KEY", "20250814212211uibrf84hdbkrc4sa88jrobb9me")  # ← 실제키/환경변수
BASE_URL  = "https://stcis.go.kr/openapi/busroutesttn.json"

ROUTES_CSV   = "bus_routes_all_2.csv"   # (routeId, 노선번호, [시도코드 후보열])
HIER_CSV     = "hierarchy.csv"          # (시도코드/시군(구)코드/읍면동코드)
SIDO_CSV     = "sido.csv"               # (옵션) 시도명 매핑용

OUT_CSV      = "route_stops_all.csv"    # 최종 병합 결과
PART_DIR     = "out_parts"
CACHE_PATH   = "./busroutesttn_cache.json"

# 동시성/성능
TIMEOUT_SEC  = 8
MAX_RETRY    = 2
BACKOFF_BASE = 0.4
REQ_DELAY    = 0.04
WORKERS      = 48
BATCH_SIZE   = 500
CACHE_FLUSH_EVERY = 2000

# 샘플링 상한(기본)
MAX_SGG_PER_SD_DEFAULT  = 20
MAX_EMD_PER_SGG_DEFAULT = 40
# 강원/전북(및 alias) 확대 상한
OVERRIDE_LIMITS = {
    "51": (80, 120), "42": (80, 120),  # 강원특별자치도 / 강원도
    "52": (80, 120), "45": (80, 120),  # 전북특별자치도 / 전라북도
}

# 시도코드 alias (혼재 대응)
SDCD_ALIASES: Dict[str, List[str]] = {
    "49": ["49", "50"],   # 제주
    "50": ["50", "49"],   # 제주
    "51": ["51", "42"],   # 강원특별자치도 → 강원도
    "52": ["52", "45"],   # 전북특별자치도 → 전라북도
}
def alias_list(sd: str) -> List[str]:
    sd = str(sd).zfill(2)
    return SDCD_ALIASES.get(sd, [sd])

# ----------------- 락/세션 -----------------
cache_lock = threading.Lock()
file_lock  = threading.Lock()

def make_session() -> requests.Session:
    s = requests.Session()
    if Retry is not None:
        retry = Retry(total=MAX_RETRY, backoff_factor=0.2,
                      status_forcelist=[429, 500, 502, 503, 504],
                      allowed_methods=["GET"], raise_on_status=False)
        adapter = HTTPAdapter(pool_connections=WORKERS*2, pool_maxsize=WORKERS*2, max_retries=retry)
    else:
        adapter = HTTPAdapter(pool_connections=WORKERS*2, pool_maxsize=WORKERS*2)
    s.mount("https://", adapter); s.mount("http://", adapter)
    return s

# ----------------- 공통 I/O -----------------
def read_csv_kr(path: str) -> pd.DataFrame:
    for enc in ("utf-8-sig", "utf-8", "cp949"):
        try: return pd.read_csv(path, encoding=enc)
        except Exception: pass
    return pd.read_csv(path)

def _load_cache() -> Dict[str, Dict[str, Any]]:
    if not os.path.exists(CACHE_PATH): return {}
    try:
        with open(CACHE_PATH, "r", encoding="utf-8") as f: return json.load(f)
    except Exception:
        return {}

def _save_cache(cache: Dict[str, Dict[str, Any]]) -> None:
    with cache_lock:
        snapshot = copy.deepcopy(cache)
    tmp = f"{CACHE_PATH}.tmp-{uuid.uuid4().hex}"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(snapshot, f, ensure_ascii=False)
        f.flush()
        try: os.fsync(f.fileno())
        except Exception: pass
    try:
        os.replace(tmp, CACHE_PATH)
    finally:
        try: os.remove(tmp)
        except Exception: pass

def ensure_part_dir():
    if not os.path.isdir(PART_DIR):
        os.makedirs(PART_DIR, exist_ok=True)

def write_part(rows: List[Dict[str, Any]]) -> None:
    if not rows: return
    ensure_part_dir()
    df = pd.DataFrame(rows)
    tmp = os.path.join(PART_DIR, f"part-{uuid.uuid4().hex}.csv")
    with open(tmp, "w", encoding="utf-8-sig") as f:
        df.to_csv(f, index=False)
        f.flush()
        try: os.fsync(f.fileno())
        except Exception: pass

def finalize_output():
    frames = []
    if os.path.exists(OUT_CSV):
        try: frames.append(pd.read_csv(OUT_CSV, encoding="utf-8-sig"))
        except Exception: pass
    if os.path.isdir(PART_DIR):
        for fn in os.listdir(PART_DIR):
            if fn.endswith(".csv"):
                try: frames.append(pd.read_csv(os.path.join(PART_DIR, fn), encoding="utf-8-sig"))
                except Exception: pass
    if not frames: return
    df = pd.concat(frames, ignore_index=True)
    key_cols = [c for c in ["routeId","정류장순번","정류장ID"] if c in df.columns]
    if key_cols:
        df = df.drop_duplicates(subset=key_cols, keep="first").reset_index(drop=True)
    tmp = f"{OUT_CSV}.tmp-{uuid.uuid4().hex}"
    with open(tmp, "w", encoding="utf-8-sig") as f:
        df.to_csv(f, index=False)
        f.flush()
        try: os.fsync(f.fileno())
        except Exception: pass
    os.replace(tmp, OUT_CSV)
    # 파트 비우기
    if os.path.isdir(PART_DIR):
        for fn in os.listdir(PART_DIR):
            try: os.remove(os.path.join(PART_DIR, fn))
            except Exception: pass

# ----------------- 로더(표준화) -----------------
def load_routes() -> pd.DataFrame:
    """
    ROUTES_CSV에서 routeId/routeNo/시도코드 후보열을 표준화.
    허용 컬럼:
      - routeId: routeId | ROUTEID | route_id
      - 노선번호: 노선번호 | routeNo | ROUTENO (없으면 공란)
      - 시도코드: API시도코드 | 시도코드 | 요청시도코드 | sdCd | sido_code (없으면 공란)
    """
    df = read_csv_kr(ROUTES_CSV)
    rid = next((c for c in ["routeId","ROUTEID","route_id"] if c in df.columns), None)
    if rid is None:
        raise ValueError(f"{ROUTES_CSV}에 routeId 컬럼이 필요합니다.")
    rno = next((c for c in ["노선번호","routeNo","ROUTENO"] if c in df.columns), None)
    if rno is None:
        df[rno := "노선번호"] = ""
    sd  = next((c for c in ["API시도코드","시도코드","요청시도코드","sdCd","sido_code"] if c in df.columns), None)

    out = pd.DataFrame({
        "routeId": df[rid].astype(str),
        "routeNo": df[rno].astype(str),
        "sdCd":    (df[sd].astype(str) if sd else "").astype(str) if sd else ""
    })
    if sd:
        out["sdCd"] = out["sdCd"].str.extract(r"(\d+)")[0].fillna("").astype(str).str.zfill(2)
    out = out.dropna(subset=["routeId"]).drop_duplicates(subset=["routeId"]).reset_index(drop=True)
    return out

def load_hierarchy() -> pd.DataFrame:
    """
    hierarchy.csv 표준화:
      - 시도코드(2), 시군구코드(5), 읍면동코드(10)
      - 허용 별칭: 시군코드/시군구코드/sggCd, 읍면동코드/emdCd
    """
    df = read_csv_kr(HIER_CSV)
    sd  = next((c for c in ["시도코드","sdCd"] if c in df.columns), None)
    sgg = next((c for c in ["시군구코드","시군코드","sggCd"] if c in df.columns), None)
    emd = next((c for c in ["읍면동코드","emdCd"] if c in df.columns), None)
    if sd is None:
        raise ValueError("hierarchy.csv에 (시도코드|sdCd) 컬럼이 필요합니다.")
    out = pd.DataFrame({"시도코드": df[sd].astype(str).str.extract(r"(\d+)")[0].str.zfill(2)})
    out["시군구코드"] = df[sgg].astype(str).str.extract(r"(\d+)")[0].str.zfill(5) if sgg else ""
    out["읍면동코드"] = df[emd].astype(str).str.extract(r"(\d+)")[0].str.zfill(10) if emd else ""
    out.loc[~out["시도코드"].str.fullmatch(r"\d{2}"), "시도코드"] = ""
    out.loc[~out["시군구코드"].str.fullmatch(r"\d{5}"), "시군구코드"] = ""
    out.loc[~out["읍면동코드"].str.fullmatch(r"\d{10}"), "읍면동코드"] = ""
    out = out[out["시도코드"]!=""].drop_duplicates().reset_index(drop=True)
    return out

def load_sido_map() -> Dict[str, str]:
    if not os.path.exists(SIDO_CSV): return {}
    df = read_csv_kr(SIDO_CSV)
    code = next((c for c in ["시도코드","sdCd"] if c in df.columns), None)
    name = next((c for c in ["시도명","sdNm"] if c in df.columns), None)
    if not code or not name: return {}
    return { str(r[code]).zfill(2): str(r[name]) for _, r in df.iterrows() }

def load_done_route_ids(out_csv: str) -> Set[str]:
    if not os.path.exists(out_csv): return set()
    done: Set[str] = set()
    for chunk in pd.read_csv(out_csv, encoding="utf-8-sig", chunksize=200_000, dtype=str):
        col = next((c for c in ["routeId","ROUTEID"] if c in chunk.columns), None)
        if col:
            done |= set(chunk[col].dropna().astype(str).unique().tolist())
            break
    return done

# ----------------- API -----------------
def get_json(session: requests.Session, params: Dict[str, str]) -> Dict[str, Any]:
    r = session.get(BASE_URL, params=params, timeout=TIMEOUT_SEC)
    r.raise_for_status()
    data = r.json()
    st = data.get("status")
    if st not in ("OK","NOT_FOUND"):
        raise RuntimeError(f"API status={st}: {str(data)[:200]}")
    return data

def request_busroutesttn(session: requests.Session, *,
                         sdCd: Optional[str]=None,
                         sggCd: Optional[str]=None,
                         emdCd: Optional[str]=None,
                         routeId: str) -> Dict[str, Any]:
    params = {"apikey": API_KEY, "routeId": routeId}
    if sdCd:  params["sdCd"]  = sdCd
    if sggCd: params["sggCd"] = sggCd
    if emdCd: params["emdCd"] = emdCd
    for i in range(MAX_RETRY + 1):
        try:
            data = get_json(session, params)
            time.sleep(REQ_DELAY)
            return data
        except Exception:
            if i == MAX_RETRY: raise
            time.sleep(BACKOFF_BASE*(2**i) + 0.05*i)

def norm_result(data: Dict[str, Any]) -> List[Dict[str, Any]]:
    if data.get("status") != "OK": return []
    res = data.get("result")
    if isinstance(res, dict): return [res]
    if isinstance(res, list): return res
    return []

# ----------------- 쿼리 후보 생성 -----------------
def _seed_from_route(rid: str) -> int:
    return int(hashlib.md5(rid.encode("utf-8")).hexdigest()[:8], 16)

def _shuffle_take(arr: np.ndarray, k: int, rng: np.random.RandomState) -> List[str]:
    if arr.size == 0 or k <= 0:
        return []
    idx = rng.permutation(arr.size)[:k]
    return arr[idx].tolist()

def build_trials(sd: str, rid: str, hier: pd.DataFrame) -> List[Dict[str, str]]:
    """
    한 노선(routeId)에 대해:
    1) sdCd (alias 포함)
    2) 해당(및 alias) 시도의 sggCd 섞어서 상위 N개
    3) 각 sgg의 emdCd도 섞어서 일부
    """
    trials: List[Dict[str, str]] = []
    seed = _seed_from_route(rid)
    rng = np.random.RandomState(seed)

    sds = alias_list(sd) if sd else []
    # limits 결정(강원/전북/alias는 크게)
    def _limits(s):
        return OVERRIDE_LIMITS.get(s, (MAX_SGG_PER_SD_DEFAULT, MAX_EMD_PER_SGG_DEFAULT))

    # 1) sdCd
    for s in sds:
        trials.append({"sdCd": s})

    # 2) sggCd / 3) emdCd
    base_sds = sds or list(OVERRIDE_LIMITS.keys())  # sdCd 없으면 강화 대상 풀도 탐색
    for s in base_sds:
        lim_sgg, lim_emd = _limits(s)
        sggs = hier.loc[(hier["시도코드"]==s) & (hier["시군구코드"]!=""), "시군구코드"].dropna().unique()
        sggs = _shuffle_take(np.array(sggs), lim_sgg, rng)
        trials += [{"sggCd": g} for g in sggs]

        emd_rows = hier[(hier["시도코드"]==s) & (hier["읍면동코드"]!="")]
        if not emd_rows.empty and sggs:
            for g in sggs:
                emds = emd_rows.loc[emd_rows["시군구코드"]==g, "읍면동코드"].dropna().unique()
                emds = _shuffle_take(np.array(emds), lim_emd, rng)
                trials += [{"emdCd": e} for e in emds]

    # 중복 제거
    uniq, seen = [], set()
    for t in trials:
        k = json.dumps(t, sort_keys=True, ensure_ascii=False)
        if k not in seen:
            uniq.append(t); seen.add(k)

    return uniq or [{}]

# ----------------- 1 route 처리 -----------------
def _row_from_item(it: Dict[str,Any], rid: str, rno: str, sd: str, sido_name_map: Dict[str,str]) -> Dict[str,Any]:
    sdNm = it.get("sdNm") or (sido_name_map.get(sd) if sd else "") or (it.get("sdCd") or "")
    return {
        "routeId": rid,
        "버스번호": it.get("routeNo") or rno,
        "정류장순번": it.get("sttnSeq"),
        "정류장ID": it.get("sttnId"),
        "정류장명": it.get("sttnNm"),
        "시도": sdNm,
        "시군구": it.get("sggNm"),
        "읍면동": it.get("emdNm"),
    }

def fetch_one_route(row: pd.Series,
                    hier: pd.DataFrame,
                    sido_name_map: Dict[str,str],
                    session: requests.Session,
                    cache: Dict[str, Dict[str, Any]]) -> List[Dict[str, Any]]:
    rid = str(row["routeId"])
    rno = str(row.get("routeNo",""))
    sd  = str(row.get("sdCd","") or "").zfill(2) if str(row.get("sdCd","")) else ""

    # routeId 단위 성공 응답 캐시
    key = f"RID_OK:{rid}"
    with cache_lock:
        ok_hit = cache.get(key)
    if ok_hit:
        items = norm_result(ok_hit)
        return [_row_from_item(it, rid, rno, sd, sido_name_map) for it in items]

    trials = build_trials(sd, rid, hier)
    for t in trials:
        try:
            data = request_busroutesttn(session, routeId=rid, **t)
        except Exception:
            continue
        items = norm_result(data)
        if items:
            with cache_lock:
                cache[key] = data
            return [_row_from_item(it, rid, rno, sd, sido_name_map) for it in items]
    return []

# ----------------- 메인 -----------------
def main():
    if not API_KEY or API_KEY in {"여기에_실키_넣으세요","YOUR_REAL_KEY","your_api_key_here"}:
        raise RuntimeError("STCIS_API_KEY 환경변수 또는 코드의 API_KEY에 실제 키를 지정하세요.")

    routes = load_routes()
    hier   = load_hierarchy()
    sido_map = load_sido_map()

    # 증분: OUT_CSV에 이미 있는 routeId는 스킵
    done_ids = load_done_route_ids(OUT_CSV)
    if done_ids:
        routes = routes[~routes["routeId"].isin(done_ids)].reset_index(drop=True)

    total = len(routes)
    print(f"[INFO] 대상 routeId {total}건")
    if total == 0:
        print("[INFO] 새로 조회할 노선 없음")
        return

    session = make_session()
    cache   = _load_cache()

    processed = 0
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        for start in range(0, total, BATCH_SIZE):
            end = min(start + BATCH_SIZE, total)
            batch = routes.iloc[start:end]
            futs = [ex.submit(fetch_one_route, r, hier, sido_map, session, cache) for _, r in batch.iterrows()]
            batch_rows: List[Dict[str, Any]] = []
            for fut in as_completed(futs):
                rows = fut.result()
                batch_rows.extend(rows)
                processed += 1
                if processed % CACHE_FLUSH_EVERY == 0:
                    _save_cache(cache)
                    print(f"[flush] {processed}/{total}")

            write_part(batch_rows)
            print(f"[progress] {processed}/{total} ({processed*100.0/total:.1f}%)")

    _save_cache(cache)
    finalize_output()
    print(f"[DONE] 완료 → {OUT_CSV}")

if __name__ == "__main__":
    main()


[INFO] 대상 routeId 4722건
[progress] 500/4722 (10.6%)
[progress] 1000/4722 (21.2%)
[progress] 1500/4722 (31.8%)
[flush] 2000/4722
[progress] 2000/4722 (42.4%)
[progress] 2500/4722 (52.9%)
[progress] 3000/4722 (63.5%)
[progress] 3500/4722 (74.1%)
[flush] 4000/4722
[progress] 4000/4722 (84.7%)
[progress] 4500/4722 (95.3%)
[progress] 4722/4722 (100.0%)


C:\Users\hyunj\AppData\Local\Temp\ipykernel_9672\905702792.py:126: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  try: frames.append(pd.read_csv(OUT_CSV, encoding="utf-8-sig"))


[DONE] 완료 → route_stops_all.csv


In [16]:
import pandas as pd

file_path = 'route_stops_all.csv'

df = pd.read_csv(file_path)
df = df.dropna(how='all')

df.to_csv(file_path)

C:\Users\hyunj\AppData\Local\Temp\ipykernel_8200\3559195701.py:5: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
